## Model and Custom Dataset classes

In [2]:
import cv2
import mediapipe as mp
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os

class FallDetectionLSTM(nn.Module):
    def __init__(self, input_size=66, hidden_size=64, num_layers=2, num_classes=2):
        super(FallDetectionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

class FallVideoDataset(Dataset):
    def __init__(self, video_dir, num_videos_fall=13, num_videos_normal=6, seq_length=30):
        self.seq_length = seq_length
        raw_data = []
        raw_labels = []
        
        model_path = 'mediapipe_weights/pose_landmarker_full.task'
        options = mp.tasks.vision.PoseLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
            running_mode=mp.tasks.vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5,
            num_poses=1
        )
        
        print("Extracting FALL key points")
        
        for i in range(1, num_videos_fall + 1):
            video_path = os.path.join(video_dir+'fall/', f"{i}.mp4")  
            if not os.path.exists(video_path):
                print(f"Not found: {video_path}")
                continue
                
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps <= 0: 
                fps = 30.0
                
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            landmarks_sequence = []
            frame_idx = 0
            
            with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    success, frame = cap.read()
                    if not success:
                        break
                        
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    timestamp_ms = int(frame_idx * (1000 / fps))
                    
                    try:
                        result = landmarker.detect_for_video(mp_image, timestamp_ms)
                        
                        frame_features = []
                        if result.pose_landmarks and len(result.pose_landmarks) > 0:
                            center_x = result.pose_landmarks[0][23].x
                            center_y = result.pose_landmarks[0][23].y
                            for lm in result.pose_landmarks[0]:
                                frame_features.extend([lm.x-center_x, lm.y-center_y])
                        
                        if len(frame_features) < 66:
                            frame_features = [0.0] * 66
                            
                        landmarks_sequence.append((frame_features, 1))
                        
                    except Exception as e:
                        print(f"Error on frame {frame_idx} in {i}: {e}")
                        
                    frame_idx += 1
                    
            cap.release()
            
            for idx in range(len(landmarks_sequence) - seq_length + 1):
                window = landmarks_sequence[idx : idx + seq_length]
                
                window_features = [item[0] for item in window] 
                window_label = window[-1][1]                   
                
                raw_data.append(window_features)
                raw_labels.append(window_label)


        print("Extracting NORMAL key points")
        
        for i in range(1, num_videos_normal + 1):
            video_path = os.path.join(video_dir+'no_fall/', f"{i}.mp4")  
            if not os.path.exists(video_path):
                print(f"Not found: {video_path}")
                continue
                
            cap = cv2.VideoCapture(video_path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps <= 0: 
                fps = 30.0
                
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            landmarks_sequence = []
            frame_idx = 0
            
            with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    success, frame = cap.read()
                    if not success:
                        break
                        
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    timestamp_ms = int(frame_idx * (1000 / fps))
                    
                    try:
                        result = landmarker.detect_for_video(mp_image, timestamp_ms)
                        
                        frame_features = []
                        if result.pose_landmarks and len(result.pose_landmarks) > 0:
                            center_x = result.pose_landmarks[0][23].x
                            center_y = result.pose_landmarks[0][23].y
                            for lm in result.pose_landmarks[0]:
                                frame_features.extend([lm.x-center_x, lm.y-center_y])
                        
                        if len(frame_features) < 66:
                            frame_features = [0.0] * 66
                            
                        landmarks_sequence.append((frame_features, 0))
                        
                    except Exception as e:
                        print(f"Error on frame {frame_idx} in {i}: {e}")
                        
                    frame_idx += 1
                    
            cap.release()
            
            for idx in range(len(landmarks_sequence) - seq_length + 1):
                window = landmarks_sequence[idx : idx + seq_length]
                
                window_features = [item[0] for item in window]
                window_label = window[-1][1]                 
                
                raw_data.append(window_features)
                raw_labels.append(window_label)
        if len(raw_data) == 0:
            raise RuntimeError(f"Error: {video_dir}")
            
        print(f"Success. Train sequences: {len(raw_data)}")
        
        self.data = torch.tensor(raw_data, dtype=torch.float32)
        self.labels = torch.tensor(raw_labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]



## Train loop

In [1]:
def train_model(train_dataset, INPUT_SIZE, BATCH_SIZE, EPOCHS, LR):
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    dataset = train_dataset
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = FallDetectionLSTM(input_size=INPUT_SIZE).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    
    print(f"device: {DEVICE}")
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
            
        avg_loss = epoch_loss / total
        accuracy = correct / total
        print(f"Epoch [{epoch+1}/{EPOCHS}] loss: {avg_loss:.4f} | acc: {accuracy*100:.2f}%")
        
    torch.save(model.state_dict(), 'fall_detection_lstm.pth')
    print("Weights saved: 'fall_detection_lstm.pth'")

SEQ_LENGTH = 30   
INPUT_SIZE = 66   
BATCH_SIZE = 16
EPOCHS = 20
LR = 0.001
dataset_train = FallVideoDataset(video_dir='videos/', num_videos_fall=13, num_videos_normal=8, seq_length=SEQ_LENGTH)
train_model(dataset_train, INPUT_SIZE, BATCH_SIZE, EPOCHS, LR)

NameError: name 'FallVideoDataset' is not defined

## Testing

In [17]:
null_vector = [0.0] * 66

        
def run_inference_live(video_path):
    SEQ_LENGTH = 15
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = FallDetectionLSTM(input_size=66)
    if os.path.exists('fall_detection_lstm.pth'):
        model.load_state_dict(torch.load('fall_detection_lstm.pth', map_location=DEVICE))
        print("Model weights are loaded successfully")
    else:
        print("Missing fall_detection_lstm.pth")
        return
        
    model.to(DEVICE)
    model.eval()
    
    model_path = 'mediapipe_weights/pose_landmarker_full.task'
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.8,
        num_poses=1
    )
    
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_FPS, 20)
    h, w = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    input_buffer = []
    current_state = "Normal"
    BODY_CONNECTIONS = [
        (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
        (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
        (12, 24), (24, 26), (26, 28), (28, 30), (30, 32), (32, 28),
        (11, 23), (23, 25), (25, 27), (27, 29), (29, 31), (31, 27)
    ]
    
    frame_idx = 0
    if h==w==-1:
        print("Failed to open the video")
        return
        
    current_state = "NO PERSON" 
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                print('Failed to read the frame')
                break
                
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int(frame_idx * (1000 / cap.get(cv2.CAP_PROP_FPS)))
            frame_idx += 1
            
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            
            frame_features = null_vector
            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                frame_features = []
                center_x = landmarks[23].x
                center_y = landmarks[23].y
                for lm in landmarks:
                    frame_features.extend([lm.x-center_x, lm.y-center_y])
                
                for connection in BODY_CONNECTIONS:
                    start_idx, end_idx = connection
                    pt1, pt2 = landmarks[start_idx], landmarks[end_idx]
                    if pt1.visibility > 0.5 and pt2.visibility > 0.5:
                        cv2.line(frame, (int(pt1.x * w), int(pt1.y * h)), (int(pt2.x * w), int(pt2.y * h)), (0, 255, 0), 2)
            
            input_buffer.append(frame_features)
            if len(input_buffer) > SEQ_LENGTH:
                input_buffer.pop(0)
                
            if len(input_buffer) == SEQ_LENGTH:
                seq_tensor = torch.tensor([input_buffer], dtype=torch.float32).to(DEVICE)
                with torch.no_grad():
                    output = model(seq_tensor)
                    _, pred = torch.max(output, 1)

                    
                    if null_vector == input_buffer[-1]:
                        current_state = "NO PERSON"
                    elif pred.item() == 1 and not(input_buffer[0] == null_vector):
                        current_state = "FALL DETECTED!"    
                    else:
                        current_state = "NORMAL"

                    
                    if current_state == "FALL DETECTED!":
                        color = (0, 0, 255)
                    elif current_state == "NO PERSON":
                        color = (255, 0, 0)
                    else:
                        color =(0, 255, 0)
                    cv2.putText(frame, f"STATE: {current_state}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2, cv2.LINE_AA)
                    resized_frame = cv2.resize(frame, (540, 960))
                    cv2.imshow('Mediapipe + LSTM Fall Detection', resized_frame)
                    if cv2.waitKey(10) & 0xFF == ord('q'):
                        break
    cap.release()
    cv2.destroyAllWindows()


run_inference_live(video_path='videos/fall/12_n.mp4')

Model weights are loaded successfully
